# [LEC] LangGraph 정통 이론 & 고급 기능

슬라이드와 설명, 퀴즈로 LangGraph가 왜 필요한지, 그래프를 무엇으로 조립하는지, Agent를 어떻게 안정적으로 실행하는지 익힙니다.

셀을 위에서부터 차례로 실행하면 슬라이드와 퀴즈가 나타납니다.

**목차**

- 1. LangGraph의 필요성 (슬라이드) (퀴즈)
  - 1.1 Cyclic Graph와 반복 추론 (슬라이드) (퀴즈)
  - 1.2 LangChain과 LangGraph 비교 (슬라이드) (퀴즈)
- 2. 그래프 구성 요소 (슬라이드) (퀴즈)
  - 2.1 Node, Edge, State (슬라이드) (퀴즈)
  - 2.2 State Schema: TypedDict와 Pydantic (슬라이드) (퀴즈)
  - 2.3 Reducer (슬라이드) (퀴즈)
- 3. 안정적인 Agent 실행 (슬라이드) (퀴즈)
  - 3.1 Checkpointer와 상태 영속화 (슬라이드) (퀴즈)
  - 3.2 Human-in-the-loop (슬라이드) (퀴즈)
- 4. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("2-1", [1, 2])

이번 시간은 LangGraph를 세 덩어리로 봅니다.

1. LangGraph의 필요성: 반복하는 Agent를 순환 그래프로 표현하는 이유, LangChain과의 차이
2. 그래프 구성 요소: Node(노드), Edge(엣지), State(상태), State Schema(상태의 모양 정의), Reducer(값을 합치는 규칙)
3. 안정적인 Agent 실행: Checkpointer(실행 상태 저장)와 Human-in-the-loop(사람의 판단 끼워 넣기)

이 노트북은 개념만 다룹니다.

## 1. LangGraph의 필요성 <small>(슬라이드) (퀴즈)</small>

### 1.1 Cyclic Graph와 반복 추론 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-1", [3, 4])

Agent는 모델을 한 번 호출하고 끝나지 않는 경우가 많습니다.
다음 행동을 판단하고, 도구를 쓰고, 결과를 관찰한 뒤 다시 판단합니다.

- 검색 결과가 부족하면 다시 검색합니다.
- 도구 실행이 실패하면 다른 경로를 고릅니다.
- 답변 품질이 낮으면 스스로 검토하고 보완합니다.

그림에서 "실행 결과 확인" 뒤에 "다음 행동을 추론"으로 되돌아가는 화살표(추가 판단 필요)가 있습니다.
이렇게 앞 단계로 돌아가는 흐름을 Cyclic Graph(순환 그래프)로 표현합니다.

**왜 배우나:** 1일차에 본 ReAct(추론 → 행동 → 관찰의 반복)를 코드로 만드는 틀이 순환 그래프입니다. 반복을 그래프로 그려 두면 어디서 되돌아가고 어디서 끝나는지 한눈에 보입니다.

In [ ]:
show_quiz("../assets/quizzes/2-1/q11.json")

**정리:** Agent는 판단 → 도구 사용 → 관찰을 되풀이하고, 이 반복은 순환 그래프로 표현합니다.

### 1.2 LangChain과 LangGraph 비교 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-1", [5])

두 도구는 잘하는 일이 다릅니다.

| | LangChain | LangGraph |
|---|---|---|
| 강한 곳 | LLM, 프롬프트, 도구, 파서를 잇는 체인 | 상태를 가진 반복 흐름과 분기 흐름 |
| 흐름 모양 | 입력에서 출력까지 비교적 한 방향 | 조건에 따라 다음 노드를 고르고, 되돌아가기도 함 |
| 대표 기능 | LCEL(실행 단위를 파이프 기호로 잇는 문법) | 조건부 라우팅, 중단·재개·승인·재시도 |
| 데이터 | 앞 단계의 출력이 다음 단계의 입력 | 작업 상태를 그래프 전체에서 공유 |

LangGraph는 복잡한 Agent를 추적할 수 있는 실행 그래프로 표현합니다.

**왜 배우나:** 만들려는 기능이 한 방향 파이프라인인지, 반복·분기·승인이 필요한 흐름인지 보고 도구를 고를 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/2-1/q12.json")

**정리:** 한 방향 파이프라인은 LangChain, 상태를 들고 반복·분기·중단·재개하는 흐름은 LangGraph가 강합니다.

## 2. 그래프 구성 요소 <small>(슬라이드) (퀴즈)</small>

### 2.1 Node, Edge, State <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-1", [6, 7])

LangGraph의 기본 요소는 세 가지입니다.

- **Node(노드):** 실제 작업을 하는 실행 단위입니다. 파이썬 함수 하나가 노드 하나입니다.
- **Edge(엣지):** 한 노드가 끝난 뒤 이동할 노드를 정합니다.
- **State(상태):** 그래프가 실행되는 동안 유지되는 공유 데이터입니다.

슬라이드 코드에서 `AgentState`가 State이고 `call_llm`이 Node입니다.
`call_llm`은 State를 받아서 바꿀 값(`answer`)만 담은 딕셔너리를 돌려줍니다.
`graph.add_node("llm", call_llm)`이 이 함수를 `"llm"`이라는 이름의 노드로 그래프에 올립니다.

In [ ]:
show_slides("2-1", [8])

Agent 그래프는 State를 읽고 고치는 노드들을 이어 놓은 것으로 볼 수 있습니다.

- LLM 호출 노드는 다음 행동을 판단합니다.
- Tool Node(도구 노드)는 고른 도구를 실행하고 결과를 State에 기록합니다.
- 조건부 Edge는 State를 보고 도구를 더 쓸지, 최종 답변으로 갈지 정합니다.

그림에서 "정보가 충분한가?" 뒤의 두 갈래(추가 검색 필요 / 충분함)가 조건부 Edge입니다.

**왜 배우나:** 이번 실습의 그래프는 모두 이 세 가지로만 조립합니다. 노드를 더하고, 엣지로 잇고, State로 값을 넘기는 순서를 알면 처음 보는 그래프 코드도 읽을 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/2-1/q21.json")

**정리:** Node는 일을 하고, Edge는 다음 행선지를 정하고, State는 노드들이 함께 쓰는 데이터입니다.

### 2.2 State Schema: TypedDict와 Pydantic <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-1", [9])

State Schema(상태 스키마)는 그래프가 어떤 데이터를 들고 실행되는지 정합니다.

- **TypedDict:** 가볍고 직관적입니다. 필드 이름과 타입만 적습니다.
- **Pydantic `BaseModel`:** 값 검증, 기본값, 설명이 필요할 때 유리합니다. 슬라이드처럼 `Field(description=...)`로 설명을, `default_factory=list`나 `= None`으로 기본값을 줍니다.

스키마를 분명히 정해 두면 노드끼리 주고받는 데이터의 약속이 흔들리지 않습니다.

In [ ]:
show_quiz("../assets/quizzes/2-1/q22.json")

**정리:** 가볍게 쓸 때는 TypedDict, 검증·기본값·설명이 필요하면 Pydantic으로 State를 정의합니다.

### 2.3 Reducer <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-1", [10])

Reducer(리듀서)는 여러 노드가 같은 State 필드를 고칠 때 값을 합치는 규칙입니다.
규칙이 분명해야 그래프를 실행한 결과를 예측할 수 있습니다.

- 메시지 목록은 새 메시지를 기존 목록 뒤에 쌓아야 합니다.
- 점수나 카운터는 더하거나 최신 값으로 바꿀 수 있습니다.
- 여러 노드를 동시에 실행(병렬 실행)할 때는 여러 결과를 한 State로 합치는 규칙이 특히 중요합니다.

슬라이드 코드의 `Annotated[list, add_messages]`는 "`messages` 필드는 `add_messages` 규칙으로 합친다"는 뜻입니다.

**왜 배우나:** 합치는 규칙이 없으면 노드가 돌려준 값이 기존 값을 덮어씁니다. 메시지 목록이 덮어써지면 앞 대화를 잃습니다.

In [ ]:
show_quiz("../assets/quizzes/2-1/q23.json")

**정리:** Reducer는 같은 필드에 들어온 값을 합치는 규칙이고, `add_messages`는 메시지를 쌓습니다.

## 3. 안정적인 Agent 실행 <small>(슬라이드) (퀴즈)</small>

### 3.1 Checkpointer와 상태 영속화 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-1", [11, 12])

Checkpointer(체크포인터)는 그래프의 실행 상태를 저장해 두었다가 중단한 곳부터 다시 이어 가게 하는 기능입니다.
Agent 작업이 길어지면 중간에 실패하거나, 사람의 입력이 더 필요할 때가 있습니다. 대화형 서비스에서는 사용자별 세션 상태도 유지해야 합니다.

안정적인 서비스에 필요한 이유는 세 가지입니다.

- 장애가 나면 이전 단계까지의 실행 기록을 복원합니다.
- 승인 대기, 외부 시스템 응답 대기처럼 기다렸다가 이어 가는 흐름을 다룹니다.
- 실행 이력을 추적해 디버깅과 평가에 씁니다.

**왜 배우나:** 다음 절의 사람 승인은 "멈췄다가 이어 가기"입니다. 멈춘 지점의 상태가 저장되어 있어야 이어 갈 수 있으므로 Checkpointer가 먼저 있어야 합니다.

In [ ]:
show_quiz("../assets/quizzes/2-1/q31.json")

**정리:** Checkpointer는 실행 상태를 저장해 중단·재개, 장애 복원, 실행 이력 추적을 가능하게 합니다.

### 3.2 Human-in-the-loop <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-1", [13])

Human-in-the-loop(휴먼 인 더 루프)는 Agent가 실행되는 도중 특정 지점에서 사람의 판단을 받는 설계입니다.

- 민감한 도구를 실행하기 전에 승인을 받을 수 있습니다.
- 모델의 판단이 불확실할 때 사용자에게 선택지를 물을 수 있습니다.
- 자동화의 효율과 사람의 통제권을 함께 챙깁니다.

그림에서 Action(도구 실행) 앞에 "판단 대기, 사람 승인 후 실행"이 붙어 있습니다. 승인되어야 Observation(결과 관찰)으로 넘어갑니다.

In [ ]:
show_slides("2-1", [14])

사람이 개입할 지점은 위험도와 비용을 기준으로 정합니다.

- 외부 발송, 결제, 삭제처럼 되돌리기 어려운 행동 앞에 둡니다.
- 정보가 부족하면 추측하지 않고 사람에게 묻습니다.
- 승인을 너무 자주 받으면 흐름이 느려집니다. 승인 기준을 분명히 정합니다.
- 승인 결과도 State에 기록합니다. 그래야 이후 실행이 같은 결정을 참조합니다.

**왜 배우나:** 승인 지점을 어디에 둘지는 Agent가 실제 업무에 들어갈 때 가장 먼저 부딪히는 설계 문제입니다. 거부한 결과를 State에 남기지 않으면 Agent는 거부 사실을 모른 채 멈춘 곳부터 그대로 이어 갈 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/2-1/q32.json")

**정리:** 되돌리기 어려운 행동 앞에서 사람의 승인을 받고, 승인·거부 결과도 State에 기록합니다.

## 4. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/2-1/final.json")

**이 노트북에서 본 것**
- Agent는 판단 → 도구 사용 → 관찰을 되풀이하고, 이 반복은 순환 그래프로 표현합니다.
- 한 방향 파이프라인은 LangChain, 상태를 들고 반복·분기·중단·재개하는 흐름은 LangGraph가 강합니다.
- 그래프는 Node(작업 단위), Edge(다음 이동), State(공유 데이터)로 조립합니다. 조건부 Edge는 State를 보고 행선지를 고릅니다.
- State Schema는 TypedDict로 가볍게, 검증·기본값이 필요하면 Pydantic으로 정의합니다.
- Reducer는 같은 필드의 값을 합치는 규칙이고, `add_messages`는 메시지를 쌓습니다.
- Checkpointer는 실행 상태를 저장해 중단·재개와 장애 복원을 가능하게 합니다.
- 되돌리기 어려운 행동 앞에서는 사람의 승인을 받고, 승인·거부 결과도 State에 기록합니다.